# Multimodal Cancer Classification Challenge 2026 — v17

**v11 LB = 0.6479 (high water mark). v15 LB = 0.572. v16 LB = 0.503.** Every sophistication since v11 (contrastive SSL, LOPO, EMA, aux loss, multi-snapshot) has hurt the LB. v17 strips all of that out and adds **only three robust improvements** that use no external inputs — just the competition data.

## What v17 is

- **v11 architecture verbatim:** two-branch ResNet-18, concat fusion head, mixup α=0.1, dropout 0.3, AdamW + OneCycleLR @ LR=3e-4, patient-balanced sampler, paired D4 aug.
- **v11 CV verbatim:** 3-fold StratifiedGroupKFold × 2 seeds + 1 full-data model = 7 ensemble members.

## What v17 adds (three free improvements)

1. **Stain normalization to test pixel statistics.** Compute BF/FL pixel mean and std from the test images at runtime, then normalize BOTH train and test using those stats. Bridges any intensity-distribution shift between train and test. Free, uses only the dataset.
2. **AdaBN at inference.** Before running predictions, do a single forward pass over the test set with each model in `.train()` mode (no gradients) to update BatchNorm running statistics to the test distribution. Empirically gives +0.02–0.05 on OOD problems. Free, uses only the dataset.
3. **Multi-scale TTA.** At inference, evaluate each cell at three scales (112, 128, 144 px) under all 8 D4 rotations/flips. 24-way TTA total. Robust, well-established. Costs ~2× inference time vs v11's 8-way TTA.

All three can be disabled via config flags for ablation.

## Compute budget on T4

| Stage | Time |
|---|---|
| JPEG cache (one-time) | ~45 min |
| Compute test pixel stats | ~30 s |
| 3-fold × 2 seeds supervised (10 epochs each, early stop) | ~75 min |
| Full-data model | ~15 min |
| 24-way TTA inference, 7 ckpts (with AdaBN pre-pass) | ~80 min |
| **Total** | **~3h 35min** |

## Before running

1. Attach the `rafaelproena/a3-adl` dataset (contains BF/, FL/, train.csv, sampleSubmission.csv).
2. Settings: Accelerator = `GPU T4 ×2`, Internet = `Off` (we don't need it).
3. Save Version → Save & Run All.

In [ ]:
import os
os.environ["PYTHONUNBUFFERED"] = "1"

import re, io, json, time, random, glob, functools, gc
from pathlib import Path

print = functools.partial(print, flush=True)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision import models
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from PIL import Image
import matplotlib.pyplot as plt

print("torch:", torch.__version__, "cuda:", torch.cuda.is_available(),
      "n_gpu:", torch.cuda.device_count())
!nvidia-smi -L

In [ ]:
# === DATA ROOT ===
# v11 used /kaggle/input/datasets/rafaelproena/a3-adl and scored 0.6479. v15/v16
# used the official competition path and scored worse. Sticking with what worked.
DATA_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/rafaelproena/a3-adl"),
    Path("/kaggle/input/a3-adl"),
    Path("/kaggle/input/competitions/multimodal-cancer-classification-challenge-2026"),
]
DATA_ROOT = next((p for p in DATA_ROOT_CANDIDATES if (p / "train.csv").exists()), None)
assert DATA_ROOT is not None, f"train.csv not found at any of {DATA_ROOT_CANDIDATES}"
print("DATA_ROOT =", DATA_ROOT)

OUT_DIR = Path("/kaggle/working/runs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# === IMPROVEMENTS (all default ON, can disable individually) ===
USE_TEST_STAIN_NORM = True   # compute BF/FL stats from test images, use for normalization
USE_ADABN           = True   # update BN running stats on test before inference
USE_MULTISCALE_TTA  = True   # 24-way TTA (D4 × 3 scales) instead of 8-way (D4 × 1 scale)
TTA_SCALES          = (112, 128, 144)  # used only when USE_MULTISCALE_TTA is True

# === CV (v11 settings) ===
N_SPLITS    = 3
BASE_SEED   = 1
SEEDS       = [1, 2]

# === Optimization (v11 settings) ===
EPOCHS      = 10
PATIENCE    = 5
BATCH_SIZE  = 128
LR          = 3e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP   = 1.0
MIXUP_ALPHA = 0.1
DROPOUT     = 0.3

# === Sampler (v11 settings) ===
NUM_WORKERS = 2
PATIENTS_PER_BATCH = 4

TRAIN_FULL_DATA_MODEL = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    torch.cuda.set_device(0)

# === Normalization defaults (overwritten at runtime if USE_TEST_STAIN_NORM=True) ===
# These are v11's hardcoded stats, used as a fallback if stain norm is disabled.
BF_MEAN, BF_STD = 0.504, 0.216
FL_MEAN, FL_STD = 0.100, 0.144

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

seed_everything(BASE_SEED)

In [ ]:
PAT_RE = re.compile(r"^pat_(\d+)_image_\d+\.jpg$")

def parse_patient_id(filename):
    m = PAT_RE.match(Path(filename).name)
    return int(m.group(1)) if m else None

def load_train_df(path):
    df = pd.read_csv(path); df.columns = [c.strip() for c in df.columns]
    df["patient_id"] = df["Name"].map(parse_patient_id).astype(int)
    return df

def load_test_df(path):
    df = pd.read_csv(path); df.columns = [c.strip() for c in df.columns]
    return df

def cache_split(names, bf_dir, fl_dir, label=""):
    bf_dir, fl_dir = Path(bf_dir), Path(fl_dir)
    bf, fl = {}, {}
    t0 = time.time()
    for i, n in enumerate(names):
        with open(bf_dir / n, "rb") as f: bf[n] = f.read()
        with open(fl_dir / n, "rb") as f: fl[n] = f.read()
        if (i + 1) % 20000 == 0:
            print(f"  [{label}] cached {i+1}/{len(names)} in {time.time()-t0:.1f}s")
    print(f"  [{label}] cached {len(names)} in {time.time()-t0:.1f}s")
    return bf, fl

class CachedCellDataset(Dataset):
    """v11-style dataset: reads BF + FL from RAM caches and applies transforms."""
    def __init__(self, df, bf_cache, fl_cache, bf_tf, fl_tf, paired_tf=None):
        self.df = df.reset_index(drop=True)
        self.bf_cache = bf_cache; self.fl_cache = fl_cache
        self.bf_tf = bf_tf; self.fl_tf = fl_tf
        self.paired_tf = paired_tf
    def __len__(self): return len(self.df)
    @staticmethod
    def _decode(buf): return Image.open(io.BytesIO(buf)).convert("L")
    def __getitem__(self, idx):
        row = self.df.iloc[idx]; name = row["Name"]
        bf = self.bf_tf(self._decode(self.bf_cache[name]))
        fl = self.fl_tf(self._decode(self.fl_cache[name]))
        if self.paired_tf is not None:
            bf, fl = self.paired_tf(bf, fl)
        label = int(row["Diagnosis"]) if "Diagnosis" in row else -1
        return {"bf": bf, "fl": fl, "label": label, "name": name}

In [ ]:
def stratified_patient_kfold(df, n_splits=3, seed=1):
    """v11's 3-fold StratifiedGroupKFold. seed=1 is known good (every fold has
    both classes)."""
    skgf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    y = df["Diagnosis"].to_numpy(); groups = df["patient_id"].to_numpy()
    splits = list(skgf.split(df, y=y, groups=groups))
    for f, (_, va) in enumerate(splits):
        if len(np.unique(y[va])) < 2:
            raise ValueError(f"Fold {f} has only one class — try a different seed.")
    return splits

def summarize_split(df, tr, va):
    trd, vad = df.iloc[tr], df.iloc[va]
    return (f"train: {len(tr):>6} cells, {trd['patient_id'].nunique():>2}p, "
            f"pos {trd['Diagnosis'].mean():.3f} | "
            f"val: {len(va):>6} cells, {vad['patient_id'].nunique():>2}p, "
            f"pos {vad['Diagnosis'].mean():.3f} | "
            f"val pats: {sorted(vad['patient_id'].unique().tolist())}")

class PatientBalancedSampler(Sampler):
    """Each batch contains cells from `patients_per_batch` distinct patients."""
    def __init__(self, df, batch_size, patients_per_batch=4, seed=0):
        assert batch_size % patients_per_batch == 0
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.per_pat = batch_size // patients_per_batch
        self.patients_per_batch = patients_per_batch
        self.rng = np.random.default_rng(seed)
        self.by_pat = {p: np.array(g.index.tolist())
                       for p, g in self.df.groupby("patient_id")}
        self.patients = list(self.by_pat.keys())
        self.epoch_len = len(self.df) // batch_size * batch_size
    def __len__(self): return self.epoch_len
    def __iter__(self):
        out = []
        for _ in range(self.epoch_len // self.batch_size):
            pats = self.rng.choice(self.patients,
                                   size=min(self.patients_per_batch, len(self.patients)),
                                   replace=False)
            for p in pats:
                idxs = self.by_pat[p]
                out.extend(self.rng.choice(idxs, size=self.per_pat,
                                           replace=len(idxs) < self.per_pat).tolist())
        return iter(out)

In [ ]:
def _make_resnet18_branch(pretrained=True):
    weights = "DEFAULT" if pretrained else None
    net = models.resnet18(weights=weights)
    w = net.conv1.weight.data
    new_conv = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    if pretrained:
        new_conv.weight.data = w.mean(dim=1, keepdim=True)
    net.conv1 = new_conv
    fd = net.fc.in_features; net.fc = nn.Identity()
    return net, fd  # 512

class MultimodalClassifier(nn.Module):
    """v11 architecture: two ResNet-18 branches + concat fusion head."""
    def __init__(self, pretrained=True, dropout=DROPOUT):
        super().__init__()
        self.bf_branch, fd = _make_resnet18_branch(pretrained)
        self.fl_branch, _  = _make_resnet18_branch(pretrained)
        self.head = nn.Sequential(
            nn.Linear(fd * 2, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, 1),
        )
    def forward(self, bf, fl):
        feat = torch.cat([self.bf_branch(bf), self.fl_branch(fl)], dim=1)
        return self.head(feat).squeeze(-1)

with torch.no_grad():
    _m = MultimodalClassifier(pretrained=False).cpu()
    _x = torch.zeros(2, 1, 128, 128)
    print("Output shape:", _m(_x, _x).shape,
          "  params:", sum(p.numel() for p in _m.parameters()) // 1_000_000, "M")
    del _m, _x

In [ ]:
df_train = load_train_df(DATA_ROOT / "train.csv")
df_test  = load_test_df(DATA_ROOT / "sampleSubmission.csv")
print(f"Train: {len(df_train)} cells, {df_train['patient_id'].nunique()} patients, "
      f"pos rate {df_train['Diagnosis'].mean():.4f}")
print(f"Test:  {len(df_test)} cells")

print("\n--- Caching JPEG bytes into RAM ---")
bf_train_cache, fl_train_cache = cache_split(
    df_train["Name"].tolist(),
    DATA_ROOT / "BF" / "train", DATA_ROOT / "FL" / "train", label="train")
bf_test_cache, fl_test_cache = cache_split(
    df_test["Name"].tolist(),
    DATA_ROOT / "BF" / "test", DATA_ROOT / "FL" / "test", label="test")
approx_mb = (sum(len(b) for b in bf_train_cache.values()) +
             sum(len(b) for b in fl_train_cache.values()) +
             sum(len(b) for b in bf_test_cache.values()) +
             sum(len(b) for b in fl_test_cache.values())) / (1024 * 1024)
print(f"\nApprox RAM used by JPEG cache: {approx_mb:.0f} MB")

# === IMPROVEMENT 1: stain normalization to test pixel statistics ===
# Compute BF/FL pixel mean and std from a random sample of test images, then use
# those stats for normalization of BOTH train and test. This bridges any
# intensity-distribution shift between the two domains.
def _sample_pixel_stats(cache, names, n_sample=1500):
    rng = np.random.default_rng(42)
    sampled = rng.choice(names, size=min(n_sample, len(names)), replace=False)
    pixels = []
    for n in sampled:
        img = np.asarray(Image.open(io.BytesIO(cache[n])).convert("L"),
                         dtype=np.float32) / 255.0
        pixels.append(img.ravel())
    pixels = np.concatenate(pixels)
    return float(pixels.mean()), float(pixels.std())

if USE_TEST_STAIN_NORM:
    t0 = time.time()
    BF_MEAN_T, BF_STD_T = _sample_pixel_stats(bf_test_cache, df_test["Name"].tolist())
    FL_MEAN_T, FL_STD_T = _sample_pixel_stats(fl_test_cache, df_test["Name"].tolist())
    BF_MEAN_R, BF_STD_R = _sample_pixel_stats(bf_train_cache, df_train["Name"].tolist())
    FL_MEAN_R, FL_STD_R = _sample_pixel_stats(fl_train_cache, df_train["Name"].tolist())
    print(f"\nPixel statistics (computed in {time.time()-t0:.1f}s):")
    print(f"  BF train: mean={BF_MEAN_R:.4f} std={BF_STD_R:.4f}")
    print(f"  BF test:  mean={BF_MEAN_T:.4f} std={BF_STD_T:.4f}")
    print(f"  FL train: mean={FL_MEAN_R:.4f} std={FL_STD_R:.4f}")
    print(f"  FL test:  mean={FL_MEAN_T:.4f} std={FL_STD_T:.4f}")
    # Overwrite normalization stats. Both train and test will be normalized to
    # the test distribution from this point on.
    BF_MEAN, BF_STD = BF_MEAN_T, BF_STD_T
    FL_MEAN, FL_STD = FL_MEAN_T, FL_STD_T
    print(f"  -> using TEST stats for normalization (USE_TEST_STAIN_NORM=True)")
else:
    print(f"\nUsing v11 hardcoded normalization stats (USE_TEST_STAIN_NORM=False)")

In [ ]:
# Build the normalization transforms AFTER pixel stats have been computed.
def _to_tensor_norm(mean, std):
    def fn(img):
        t = TF.to_tensor(img)
        return TF.normalize(t, [mean], [std])
    return fn

to_tensor_bf = _to_tensor_norm(BF_MEAN, BF_STD)
to_tensor_fl = _to_tensor_norm(FL_MEAN, FL_STD)

class PairedGeoAug:
    """D4 + mild rotation, applied identically to BF and FL."""
    def __init__(self, p_hflip=0.5, p_vflip=0.5, rot90=True, max_rot=10.0):
        self.p_hflip = p_hflip; self.p_vflip = p_vflip
        self.rot90 = rot90; self.max_rot = max_rot
    def __call__(self, bf, fl):
        if self.rot90:
            k = random.randint(0, 3)
            if k:
                bf = torch.rot90(bf, k, dims=(-2, -1))
                fl = torch.rot90(fl, k, dims=(-2, -1))
        if random.random() < self.p_hflip: bf, fl = TF.hflip(bf), TF.hflip(fl)
        if random.random() < self.p_vflip: bf, fl = TF.vflip(bf), TF.vflip(fl)
        if self.max_rot > 0:
            a = random.uniform(-self.max_rot, self.max_rot)
            bf, fl = TF.rotate(bf, a), TF.rotate(fl, a)
        return bf, fl

def train_modality_transform(modality):
    norm = to_tensor_bf if modality == "bf" else to_tensor_fl
    return T.Compose([T.ColorJitter(brightness=0.2, contrast=0.2), norm])

def eval_modality_transform(modality):
    return to_tensor_bf if modality == "bf" else to_tensor_fl

In [ ]:
def mixup_batch(bf, fl, y, alpha=0.1):
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(bf.size(0), device=bf.device)
    return (lam * bf + (1 - lam) * bf[idx],
            lam * fl + (1 - lam) * fl[idx],
            lam * y  + (1 - lam) * y[idx])

def run_epoch(model, loader, optimizer, scaler, criterion, train,
              mixup_alpha=0.0, grad_clip=0.0, sched=None, log_every=0):
    """v11 epoch loop. No sample weighting, no aux loss."""
    model.train(train)
    losses, hard_ys, ps = [], [], []
    t_last = time.time()
    for i, batch in enumerate(loader):
        bf = batch["bf"].to(DEVICE, non_blocking=True)
        fl = batch["fl"].to(DEVICE, non_blocking=True)
        y  = batch["label"].float().to(DEVICE, non_blocking=True)
        hard_ys.append(batch["label"].numpy())
        if train and mixup_alpha > 0:
            bf, fl, y = mixup_batch(bf, fl, y, mixup_alpha)
        with torch.amp.autocast("cuda", enabled=scaler is not None):
            logits = model(bf, fl)
            loss = criterion(logits, y)
        if train:
            optimizer.zero_grad(set_to_none=True)
            if scaler is not None:
                scaler.scale(loss).backward()
                if grad_clip > 0:
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                old_scale = scaler.get_scale()
                scaler.step(optimizer); scaler.update()
                if sched is not None and scaler.get_scale() >= old_scale:
                    sched.step()
            else:
                loss.backward()
                if grad_clip > 0: nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()
                if sched is not None: sched.step()
        losses.append(loss.item())
        ps.append(torch.sigmoid(logits).detach().float().cpu().numpy())
        if log_every and (i + 1) % log_every == 0:
            dt = time.time() - t_last; t_last = time.time()
            print(f"    step {i+1}/{len(loader)} | {dt:.1f}s | loss {float(np.mean(losses[-log_every:])):.4f}")
    hard_ys = np.concatenate(hard_ys); ps = np.concatenate(ps)
    auc = roc_auc_score(hard_ys, ps) if len(np.unique(hard_ys)) > 1 else float("nan")
    return float(np.mean(losses)), auc, hard_ys, ps

def train_one_model(train_df, val_df, ckpt_path, oof_path, hist_path,
                   epochs, seed, sampler_kind="patient", track_oof=True):
    """v11 training routine. Best-val-AUC checkpointing, optional early stop."""
    seed_everything(seed)
    train_ds = CachedCellDataset(train_df, bf_train_cache, fl_train_cache,
                                 train_modality_transform("bf"),
                                 train_modality_transform("fl"),
                                 paired_tf=PairedGeoAug())
    if sampler_kind == "patient":
        sampler = PatientBalancedSampler(train_df, batch_size=BATCH_SIZE,
                                         patients_per_batch=PATIENTS_PER_BATCH, seed=seed)
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                                  num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    else:
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                                  num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader = None
    if val_df is not None:
        val_ds = CachedCellDataset(val_df, bf_train_cache, fl_train_cache,
                                   eval_modality_transform("bf"),
                                   eval_modality_transform("fl"))
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
                                num_workers=NUM_WORKERS, pin_memory=True)

    model = MultimodalClassifier(pretrained=True, dropout=DROPOUT).to(DEVICE)
    pos = (train_df["Diagnosis"] == 1).sum()
    neg = (train_df["Diagnosis"] == 0).sum()
    pos_weight = torch.tensor(neg / max(pos, 1), device=DEVICE)
    print(f"  pos_weight={pos_weight.item():.3f}  seed={seed}  epochs={epochs}")
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=LR, steps_per_epoch=len(train_loader),
        epochs=epochs, pct_start=0.1)
    scaler = torch.amp.GradScaler("cuda") if DEVICE == "cuda" else None

    history, best_auc, best_ep, no_improve = [], -1.0, 0, 0
    for ep in range(epochs):
        t0 = time.time()
        tr_loss, tr_auc, _, _ = run_epoch(
            model, train_loader, optimizer, scaler, criterion, True,
            mixup_alpha=MIXUP_ALPHA, grad_clip=GRAD_CLIP, sched=sched, log_every=200)
        va_loss = va_auc = float("nan"); vy = vp = None
        if val_loader is not None:
            with torch.no_grad():
                va_loss, va_auc, vy, vp = run_epoch(
                    model, val_loader, None, None, criterion, False)
        dt = time.time() - t0
        print(f"  ep {ep:>2d} | tr_loss {tr_loss:.4f} tr_auc {tr_auc:.4f} "
              f"| va_loss {va_loss:.4f} va_auc {va_auc:.4f} | {dt:.1f}s")
        history.append({"epoch": ep, "tr_loss": tr_loss, "tr_auc": tr_auc,
                        "va_loss": va_loss, "va_auc": va_auc, "time": dt})
        save_now = False
        if val_loader is not None:
            if va_auc > best_auc:
                best_auc, best_ep, no_improve = va_auc, ep, 0; save_now = True
            else:
                no_improve += 1
        else:
            save_now = True; best_ep = ep
        if save_now:
            torch.save({"model": model.state_dict(), "epoch": ep,
                        "val_auc": va_auc if val_loader is not None else None,
                        "args": {"dropout": DROPOUT}}, ckpt_path)
            if track_oof and val_loader is not None:
                pd.DataFrame({"Name": val_df["Name"].values,
                              "patient_id": val_df["patient_id"].values,
                              "y_true": vy, "y_pred": vp}).to_csv(oof_path, index=False)
        if val_loader is not None and no_improve >= PATIENCE:
            print(f"  Early stopping at epoch {ep}"); break
    with open(hist_path, "w") as f:
        json.dump({"history": history, "best_auc": best_auc, "best_ep": best_ep}, f, indent=2)
    del model, optimizer, sched, scaler, train_loader
    if val_loader is not None: del val_loader
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return best_auc, best_ep

In [ ]:
print("=== Stage 1: 3-fold CV × 2 seeds (v11 reproduction) ===")
splits = stratified_patient_kfold(df_train, N_SPLITS, seed=BASE_SEED)
print(f"Folds: {len(splits)}  seeds: {SEEDS}\n")

all_results = []
for fold, (tr, va) in enumerate(splits):
    print(f"=== FOLD {fold} ===")
    print("  " + summarize_split(df_train, tr, va))
    train_df = df_train.iloc[tr].reset_index(drop=True)
    val_df   = df_train.iloc[va].reset_index(drop=True)
    for seed in SEEDS:
        tag = f"fold{fold}_seed{seed}"
        print(f"  --- {tag} ---")
        ckpt = OUT_DIR / f"{tag}_best.pt"
        oof  = OUT_DIR / f"{tag}_oof.csv"
        hist = OUT_DIR / f"{tag}_history.json"
        if ckpt.exists():
            print(f"  (already trained, skipping) {ckpt.name}\n"); continue
        best_auc, best_ep = train_one_model(
            train_df, val_df, ckpt, oof, hist,
            epochs=EPOCHS, seed=seed)
        all_results.append({"fold": fold, "seed": seed,
                            "best_auc": best_auc, "best_ep": best_ep})
        print(f"  {tag}: best AUC = {best_auc:.4f} at ep {best_ep}\n")

if all_results:
    df_results = pd.DataFrame(all_results)
    print("\n=== CV summary ===")
    print(df_results.to_string(index=False))
    print(f"Mean best AUC: {df_results['best_auc'].mean():.4f}  std {df_results['best_auc'].std():.4f}")
    print(f"Median best epoch: {int(df_results['best_ep'].median())}")

In [ ]:
if TRAIN_FULL_DATA_MODEL:
    cv_eps = []
    for hp in sorted(glob.glob(str(OUT_DIR / "fold*_history.json"))):
        h = json.load(open(hp))
        if h.get("best_ep") is not None: cv_eps.append(h["best_ep"])
    median_ep = int(np.median(cv_eps)) + 1 if cv_eps else 6
    full_epochs = max(median_ep, 5)
    print(f"\n=== FULL-DATA MODEL (epochs={full_epochs}) ===")
    ckpt = OUT_DIR / "fulldata_best.pt"
    hist = OUT_DIR / "fulldata_history.json"
    if ckpt.exists():
        print("  (already trained, skipping)")
    else:
        train_one_model(df_train, None, ckpt, None, hist,
                        epochs=full_epochs, seed=BASE_SEED + 100,
                        sampler_kind="patient", track_oof=False)
        print("  full-data model saved.")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for hp in sorted(glob.glob(str(OUT_DIR / "fold*_history.json"))):
    h = json.load(open(hp))["history"]
    label = Path(hp).stem.replace("_history", "")
    ax[0].plot([e["epoch"] for e in h], [e["va_loss"] for e in h], marker="o", label=label)
    ax[1].plot([e["epoch"] for e in h], [e["va_auc"]  for e in h], marker="o", label=label)
ax[0].set(title="Validation loss", xlabel="epoch", ylabel="BCE")
ax[1].set(title="Validation AUC",  xlabel="epoch", ylabel="AUC")
ax[1].axhline(0.85, color="red", linestyle="--", alpha=0.5, label="target 0.85")
for a in ax: a.legend(fontsize=8); a.grid(True)
plt.tight_layout()
plt.savefig("/kaggle/working/learning_curves.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
fold_oofs = []
for fold in range(N_SPLITS):
    seed_oofs = []
    for seed in SEEDS:
        p = OUT_DIR / f"fold{fold}_seed{seed}_oof.csv"
        if p.exists(): seed_oofs.append(pd.read_csv(p))
    if not seed_oofs: continue
    base = seed_oofs[0][["Name", "patient_id", "y_true"]].copy()
    base["y_pred"] = np.mean([d["y_pred"].values for d in seed_oofs], axis=0)
    fold_oofs.append(base)
if fold_oofs:
    oof = pd.concat(fold_oofs, ignore_index=True)
    cell_auc = roc_auc_score(oof["y_true"], oof["y_pred"])
    pp = oof.groupby("patient_id").agg(
        mean_pred=("y_pred", "mean"), median_pred=("y_pred", "median"),
        label=("y_true", "first")).sort_values("mean_pred")
    pat_auc = roc_auc_score(pp["label"], pp["mean_pred"])
    print("=== OOF (seed-averaged) ===")
    print(pp.to_string())
    print(f"\ncell-level OOF AUC: {cell_auc:.4f}    patient-level AUC: {pat_auc:.4f}")

In [ ]:
# === IMPROVEMENT 2: AdaBN ===
# Before predicting with a checkpoint, run test data through the model in
# train() mode (no gradients) to update BatchNorm running statistics to the
# test distribution. This is a one-pass technique with no hyperparameters.
@torch.no_grad()
def adabn_pass(model, loader):
    """Update BN running mean/var to test distribution by running test in train mode."""
    model.train()
    for batch in loader:
        bf = batch["bf"].to(DEVICE, non_blocking=True)
        fl = batch["fl"].to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=DEVICE == "cuda"):
            _ = model(bf, fl)
    model.eval()

# === IMPROVEMENT 3: multi-scale TTA ===
def _d4_at_scale(bf, fl, scale=None):
    """Yield 8 D4 augs of (bf, fl), optionally resized to `scale` first."""
    if scale is not None and scale != bf.shape[-1]:
        bf = F.interpolate(bf, size=(scale, scale), mode="bilinear", align_corners=False)
        fl = F.interpolate(fl, size=(scale, scale), mode="bilinear", align_corners=False)
    for k in range(4):
        bfr = torch.rot90(bf, k, dims=(-2, -1))
        flr = torch.rot90(fl, k, dims=(-2, -1))
        yield bfr, flr
        yield TF.hflip(bfr), TF.hflip(flr)

def _multiscale_tta(bf, fl, scales):
    """Yield all D4 augs at every scale: 8 * len(scales) total."""
    for s in scales:
        for bf_t, fl_t in _d4_at_scale(bf, fl, scale=s):
            yield bf_t, fl_t

def load_model_from_ckpt(ckpt_path):
    state = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    args = state.get("args", {})
    dropout = float(args.get("dropout", DROPOUT))
    model = MultimodalClassifier(pretrained=False, dropout=dropout).to(DEVICE)
    model.load_state_dict(state["model"]); model.eval()
    return model

def predict_one_ckpt(ckpt_path, loader, tta_scales=None):
    """Load ckpt, optional AdaBN pre-pass, then TTA inference.
    tta_scales=None means 8-way D4 at native scale (v11). Pass a tuple for multi-scale."""
    model = load_model_from_ckpt(ckpt_path)
    if USE_ADABN:
        adabn_pass(model, loader)
    aug_fn = (lambda bf, fl: _multiscale_tta(bf, fl, tta_scales)) if tta_scales \
             else (lambda bf, fl: _d4_at_scale(bf, fl))
    n_aug = 8 * (len(tta_scales) if tta_scales else 1)
    preds = []
    with torch.no_grad():
        for batch in loader:
            bf = batch["bf"].to(DEVICE, non_blocking=True)
            fl = batch["fl"].to(DEVICE, non_blocking=True)
            p = None
            for bf_t, fl_t in aug_fn(bf, fl):
                with torch.amp.autocast("cuda", enabled=DEVICE == "cuda"):
                    pi = torch.sigmoid(model(bf_t, fl_t)).float()
                p = pi if p is None else p + pi
            preds.append((p / n_aug).cpu().numpy())
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return np.concatenate(preds)

test_ds = CachedCellDataset(df_test, bf_test_cache, fl_test_cache,
                            eval_modality_transform("bf"),
                            eval_modality_transform("fl"))
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

ckpts = sorted(glob.glob(str(OUT_DIR / "fold*_best.pt")))
if TRAIN_FULL_DATA_MODEL and (OUT_DIR / "fulldata_best.pt").exists():
    ckpts.append(str(OUT_DIR / "fulldata_best.pt"))
scales_to_use = TTA_SCALES if USE_MULTISCALE_TTA else None
n_aug_total = 8 * (len(TTA_SCALES) if USE_MULTISCALE_TTA else 1)
print(f"Ensembling {len(ckpts)} models with {n_aug_total}-way TTA "
      f"(scales={scales_to_use or 'native'}, AdaBN={USE_ADABN}):")
for c in ckpts: print("  -", Path(c).name)

all_preds = []
for c in ckpts:
    t0 = time.time()
    all_preds.append(predict_one_ckpt(c, test_loader, tta_scales=scales_to_use))
    print(f"  {Path(c).name} done in {time.time()-t0:.1f}s")
preds = np.mean(all_preds, axis=0)

sub = pd.DataFrame({"Name": df_test["Name"].values, "Diagnosis": preds})
sub.to_csv("/kaggle/working/submission.csv", index=False)
print(f"\nWrote submission.csv  (mean pred = {preds.mean():.3f}, "
      f"min {preds.min():.3f}, max {preds.max():.3f})")
print(sub.head())
!wc -l /kaggle/working/submission.csv